In [ ]:
import logging
import warnings
import torch
import torchvision.transforms as transforms
import importlib
from components import FL_sim
from components.other_utilities import models_to_train
import components.broadcast_components.broadcasting_process.broadcast_reporting_utilities
import components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol
import components.broadcast_components.compressor.rans_coding
import components.other_utilities.datasets

importlib.reload(components.broadcast_components.compressor.rans_coding)
importlib.reload(components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol)
importlib.reload(components.broadcast_components.WZ_models.wz_quant_RNN)
importlib.reload(components.broadcast_components.broadcasting_process.broadcast_reporting_utilities)
importlib.reload(FL_sim)
importlib.reload(models_to_train)

from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator
from components.other_utilities.datasets import FasterSVHN
from components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol import WZServerTrainingPerRoundProtocol
from components.broadcast_components.broadcasting_process.broadcast_reporting_utilities import BroadcastMetricGatheringUtilities
from components.broadcast_components.WZ_models.wz_quant_RNN import PL_EncoderDecoder_RNN

torch.set_float32_matmul_precision('high')
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", "LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]")
warnings.filterwarnings("ignore", "You defined a `validation_step` but")
warnings.filterwarnings("ignore", "Starting from v1.9.0, `tensorboardX` has been")
warnings.filterwarnings("ignore", "The number of training batches")
warnings.filterwarnings("ignore", "`Trainer.fit` stopped: ")

In [ ]:
dataset = [
    FasterSVHN(
        # limit_count=1_500,
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

# for i in range(10):
#     for d in dataset:
#         d.labels[i]=i

In [ ]:
from components.broadcast_components.WZ_models.WZQuantizerWithDataPrep import QuantizerWithDataPrep

worker_count = 2
batch_size = 500

# *****************

user_logger = None#UnifiedLoggingClass(worker_count)
# ****
wz_model = PL_EncoderDecoder_RNN(inp_dim=1, side_info_size=0, num_planes=2,
                                 bins_per_plane=16, lr=1e-3, marginal=True).to(torch.float32)
path_to_basic = r'data/basicRNN_2plane_4bins_state.pt'
wz_model.load_state_dict(torch.load(path_to_basic, map_location='cpu'))
# ****
base_quantizer = QuantizerWithDataPrep(wz_model, train_sample_size=100_000,
        count_side_info_data=0, enable_progress_bar=False, user_logger=user_logger)
broadcast_prot_base = WZServerTrainingPerRoundProtocol(worker_count, base_quantizer, no_global_quantization=True)
broadcast_prot = BroadcastMetricGatheringUtilities(broadcast_prot_base, user_logger=user_logger)

# *****************

model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.005,)
model.load_state_dict(torch.load('data/resnet18_svhn.pth', map_location='cpu'))

# *****************
sim = FLSimulator(
    pl_model=model, num_agents=worker_count, communication_rounds=50, client_epochs_per_round=10,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False, user_logger=user_logger)
# ****
sim.run_simulation(broadcast_prot)